# Card-name OCR v1 — Kaggle training

Trains the v1 CRNN against the synthetic name-region pool produced by
`scripts/render_name_pool.py`. Mirrors `scripts/train_name_v1.py` but with
Kaggle-compatible paths and a config override layer.

## Prerequisites — upload these as Kaggle datasets first

1. **Source code** — the `moxify_ocr_model` repo (at least `src/`, `scripts/`,
   `configs/`). Upload as a Kaggle dataset; default expected name below is
   `moxify-ocr-source`.
2. **Rendered pool** — the contents of `data/synth_names/v1/` (images + jsonl).
   Upload as a Kaggle dataset; default expected name below is
   `moxify-ocr-name-pool-v1`.

Then attach BOTH datasets to this notebook (Add Data → Your datasets) and
**enable GPU + Internet** in the notebook settings.

If you named your datasets differently, edit the `SOURCE_DATASET` and
`POOL_DATASET` constants in the first code cell.

In [ ]:
# Adjust these to match your Kaggle dataset slugs.
SOURCE_DATASET = "moxify-ocr-source"        # Kaggle dataset containing src/, scripts/, configs/
POOL_DATASET = "moxify-ocr-name-pool-v1"    # Kaggle dataset containing images/ + labels.jsonl

# Where the trained model + logs land. /kaggle/working/ is the only writable path on a Kaggle kernel.
OUTPUT_DIR = "/kaggle/working/name_v1"

# Override training knobs without modifying the YAML in the source dataset.
CONFIG_OVERRIDES = {
    # 'train.epochs': 25,
    # 'train.lr': 5.0e-4,
    # 'data.batch_size': 128,
}

In [ ]:
# Confirm the inputs are mounted where we expect.
import pathlib
src_root = pathlib.Path(f"/kaggle/input/{SOURCE_DATASET}")
pool_root = pathlib.Path(f"/kaggle/input/{POOL_DATASET}")
assert src_root.exists(), f"source dataset not attached: {src_root}"
assert pool_root.exists(), f"pool dataset not attached: {pool_root}"
assert (src_root / "src" / "moxify_ocr").exists(), (
    f"expected {src_root}/src/moxify_ocr/ — did you upload the repo root?"
)
assert (pool_root / "labels.jsonl").exists(), (
    f"expected {pool_root}/labels.jsonl — did you upload the rendered pool?"
)
print("OK")
print(f"  source: {src_root} (entries: {len(list(src_root.iterdir()))})")
print(f"  pool:   {pool_root}")
print("  pool first 3 jsonl lines:")
with (pool_root / "labels.jsonl").open() as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(f"    {line.rstrip()}")

In [ ]:
# Add the source to sys.path so `import moxify_ocr` resolves.
import sys
if str(src_root / "src") not in sys.path:
    sys.path.insert(0, str(src_root / "src"))

# Sanity-check imports + GPU.
import tensorflow as tf
print("tf version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices("GPU"))
from moxify_ocr.data.name_alphabet import NAME_ALPHABET
from moxify_ocr.models.name_region import NAME_NUM_CLASSES, NAME_INPUT_SHAPE
print(f"alphabet: {len(NAME_ALPHABET)} chars  /  num_classes: {NAME_NUM_CLASSES}  /  input: {NAME_INPUT_SHAPE}")

In [ ]:
# Install anything that isn't preinstalled on Kaggle's TF kernel.
# editdistance is needed by the CER callback in scripts/train_name_v1.py.
%pip install -q editdistance pyyaml

In [ ]:
# Build a Kaggle-flavoured config by reading the repo's name_v1.yaml and
# overriding the data + train paths to point at the mounted dataset and
# /kaggle/working/.
import yaml
from pathlib import Path

base_cfg_path = src_root / "configs" / "name_v1.yaml"
with base_cfg_path.open() as f:
    cfg = yaml.safe_load(f)

cfg["data"]["pool_root"] = str(pool_root)
cfg["train"]["output_dir"] = OUTPUT_DIR

for dotted, value in CONFIG_OVERRIDES.items():
    parts = dotted.split(".")
    node = cfg
    for p in parts[:-1]:
        node = node.setdefault(p, {})
    node[parts[-1]] = value

kaggle_cfg_path = Path("/kaggle/working/name_v1_kaggle.yaml")
with kaggle_cfg_path.open("w") as f:
    yaml.safe_dump(cfg, f)
print("merged config:")
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
# Run training by invoking the existing script's main() directly. We patch
# sys.argv so its argparse picks up the Kaggle-flavoured config without
# spawning a subprocess (keeps stdout in the notebook + lets us pdb if needed).
import importlib.util
import sys

spec = importlib.util.spec_from_file_location(
    "train_name_v1", src_root / "scripts" / "train_name_v1.py"
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

old_argv = sys.argv
sys.argv = ["train_name_v1", "--config", str(kaggle_cfg_path)]
try:
    module.main()
finally:
    sys.argv = old_argv

In [ ]:
# Confirm the trained model landed in /kaggle/working/ so it's downloadable.
import pathlib
for p in sorted(pathlib.Path(OUTPUT_DIR).rglob("*")):
    print(p, p.stat().st_size if p.is_file() else "(dir)")